In [0]:
%sql
-- 1. Dimension: Surveys
CREATE OR REPLACE TABLE workspace.gold.dim_survey
CLUSTER BY (survey_id)
COMMENT 'Survey master dimension derived from Qualtrics survey IDs'
AS
SELECT DISTINCT
  survey_id,
  CASE survey_id
    WHEN 'SV_SYNTH_CUSTOMER_001' THEN 'Customer Experience & CSAT Survey'
    WHEN 'SV_SYNTH_EMPLOYEE_002' THEN 'Employee Engagement Pulse Survey'
    WHEN 'SV_SYNTH_SERVICE_003'  THEN 'Service Delivery Quality Survey'
    ELSE 'Unmapped Qualtrics Survey'
  END AS survey_name,
  CASE survey_id
    WHEN 'SV_SYNTH_CUSTOMER_001' THEN 'Customer'
    WHEN 'SV_SYNTH_EMPLOYEE_002' THEN 'Internal HR'
    WHEN 'SV_SYNTH_SERVICE_003'  THEN 'Operations'
    ELSE 'General'
  END AS survey_domain,
  source_system
FROM workspace.silver.qualtrics_responses;

-- 2. Dimension: Respondent Segment
CREATE OR REPLACE TABLE workspace.gold.dim_respondent_segment
CLUSTER BY (segment_key)
COMMENT 'Conformed respondent organizational & regional segment dimension'
AS
SELECT DISTINCT
  sha2(concat_ws('|', region, department, respondent_type), 256) AS segment_key,
  region,
  department,
  respondent_type
FROM workspace.silver.qualtrics_responses;

-- 3. Fact: Survey Responses (Grain = 1 row per response_id)
CREATE OR REPLACE TABLE workspace.gold.fact_survey_responses
CLUSTER BY (survey_id, response_date)
COMMENT 'Fact table at individual survey response grain'
AS
SELECT
  response_id,
  survey_id,
  sha2(concat_ws('|', region, department, respondent_type), 256) AS segment_key,
  CAST(recorded_date AS DATE) AS response_date,
  recorded_date,
  start_date,
  end_date,
  duration_seconds,
  ROUND(duration_seconds / 60.0, 2) AS duration_minutes,
  progress_pct,
  status,
  finished AS is_completed,
  score,
  CASE
    WHEN score IS NULL THEN 'Not Scored'
    WHEN score >= 9    THEN 'Promoter (9-10)'
    WHEN score >= 7    THEN 'Passive (7-8)'
    ELSE                    'Detractor (1-6)'
  END AS nps_bucket,
  question_1 AS satisfaction_rating,
  question_2 AS recommendation_flag,
  question_3 AS priority_level,
  _silver_processed_ts AS updated_at
FROM workspace.silver.qualtrics_responses;

-- 4. Semantic KPI View for quick validation & BI consumption
CREATE OR REPLACE VIEW workspace.gold.vw_survey_executive_kpis AS
SELECT
  s.survey_id,
  s.survey_name,
  s.survey_domain,
  seg.region,
  seg.department,
  seg.respondent_type,
  COUNT(*) AS total_responses,
  SUM(CASE WHEN f.is_completed THEN 1 ELSE 0 END) AS completed_responses,
  ROUND(SUM(CASE WHEN f.is_completed THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS completion_rate_pct,
  ROUND(AVG(f.score), 2) AS avg_score,
  ROUND(AVG(f.duration_minutes), 2) AS avg_completion_minutes,
  ROUND(
    (SUM(CASE WHEN f.score >= 9 THEN 1 ELSE 0 END) - SUM(CASE WHEN f.score <= 6 THEN 1 ELSE 0 END))
    * 100.0 / NULLIF(COUNT(f.score), 0),
    2
  ) AS net_promoter_score
FROM workspace.gold.fact_survey_responses f
JOIN workspace.gold.dim_survey s ON f.survey_id = s.survey_id
JOIN workspace.gold.dim_respondent_segment seg ON f.segment_key = seg.segment_key
GROUP BY ALL;